# Implementation Basics of Logistic Regression

## Overview
This notebook focuses on the practical aspects of implementing logistic regression. We'll cover data preprocessing, model training, evaluation metrics, and validation techniques using scikit-learn.

## Key Topics
1. Data Preprocessing and Feature Scaling
2. Model Training with scikit-learn
3. Performance Metrics (Accuracy, Precision, Recall, F1, ROC)
4. Cross-Validation Techniques
5. Hyperparameter Tuning

Let's implement these concepts with practical examples.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, classification_report, confusion_matrix

# Set random seed for reproducibility
np.random.seed(42)

## 1. Data Preprocessing and Feature Scaling

Proper data preprocessing is crucial for logistic regression as it assumes features are on similar scales. We'll use StandardScaler to standardize features by removing the mean and scaling to unit variance.

The formula for standardization is:
$$z = \frac{x - \mu}{\sigma}$$
where $\mu$ is the mean and $\sigma$ is the standard deviation of the feature.

In [ ]:
# Generate synthetic dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_redundant=5,
    n_informative=15,
    random_state=42,
    n_clusters_per_class=2
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and apply scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Visualize the effect of scaling on first two features
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='viridis')
plt.title('Before Scaling')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='viridis')
plt.title('After Scaling')
plt.xlabel('Feature 1 (scaled)')
plt.ylabel('Feature 2 (scaled)')

plt.tight_layout()
plt.show()

## 2. Model Training with scikit-learn

Let's train a logistic regression model using scikit-learn's implementation, which is optimized and includes regularization by default.

In [ ]:
# Initialize and train the model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Print model parameters
print('Model Coefficients:', model.coef_)
print('Model Intercept:', model.intercept_)

## 3. Performance Metrics

We'll evaluate our model using various metrics:
- Accuracy: Overall correctness of the model
- Precision: Proportion of positive identifications that were actually correct
- Recall: Proportion of actual positives that were identified correctly
- F1 Score: Harmonic mean of precision and recall
- ROC Curve and AUC: Receiver Operating Characteristic curve and Area Under Curve

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('Classification Report:')
print(classification_report(y_test, y_pred))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

## 4. Cross-Validation Techniques

Cross-validation helps us get a more robust estimate of model performance by splitting the data into multiple train-test sets.

In [ ]:
# Perform 5-fold cross-validation
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print('Cross-Validation Scores:', cv_scores)
print('Mean CV Score:', cv_scores.mean())
print('Standard Deviation:', cv_scores.std())

# Visualize CV scores
plt.figure(figsize=(8, 6))
plt.boxplot(cv_scores)
plt.title('5-Fold Cross-Validation Scores')
plt.ylabel('Accuracy')
plt.show()

## 5. Hyperparameter Tuning

Logistic regression has hyperparameters like regularization strength (C) and the type of solver. We'll use GridSearchCV to find the best combination.

In [ ]:
# Define parameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear']
}

# Initialize model
model = LogisticRegression(random_state=42)

# Perform grid search
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

# Print results
print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validation Score:', grid_search.best_score_)

# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)
print('Test Set Accuracy with Best Model:', accuracy_score(y_test, y_pred_best))